# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [10]:
from sklearn.preprocessing import MinMaxScaler

# --- Feature engineering / selection (igual que el lab anterior) ---
spaceship = spaceship.dropna()                       # eliminar filas con nulls
spaceship['Cabin'] = spaceship['Cabin'].str[0]       # Cabin -> letra de deck
spaceship = spaceship.drop(columns=['PassengerId', 'Name'])  # quitar identificadores
spaceship = pd.get_dummies(spaceship, drop_first=True).astype(int)  # dummies

# --- Separar features y target ---
features = spaceship.drop(columns=['Transported'])
target = spaceship['Transported']

# --- Feature Scaling ---
scaler = MinMaxScaler()
features_scaled = pd.DataFrame(
    scaler.fit_transform(features),
    columns=features.columns,
    index=features.index
)

features_scaled.head()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True
0,0.493671,0.000000,0.000000,0.000000,0.000000,0.000000,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.303797,0.010988,0.000302,0.002040,0.024500,0.002164,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,0.734177,0.004335,0.119948,0.000000,0.299670,0.002410,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
3,0.417722,0.000000,0.043035,0.030278,0.148563,0.009491,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,0.202532,0.030544,0.002348,0.012324,0.025214,0.000098,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


**Perform Train Test Split**

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    features_scaled, target, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)

(5284, 19) (1322, 19)


**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [12]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score

# BAGGING: muestreo CON reemplazo (bootstrap=True)
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    bootstrap=True,
    random_state=42
)
bagging.fit(X_train, y_train)
bagging_acc = accuracy_score(y_test, bagging.predict(X_test))
print("Bagging:", bagging_acc)

# PASTING: muestreo SIN reemplazo (bootstrap=False)
pasting = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    bootstrap=False,
    random_state=42
)
pasting.fit(X_train, y_train)
pasting_acc = accuracy_score(y_test, pasting.predict(X_test))
print("Pasting:", pasting_acc)

Bagging: 0.8040847201210287
Pasting: 0.754160363086233


- Random Forests

In [13]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_acc = accuracy_score(y_test, rf.predict(X_test))
print("Random Forest:", rf_acc)

Random Forest: 0.8055975794251135


In [14]:
importancias = pd.Series(rf.feature_importances_, index=X_train.columns)
print(importancias.sort_values(ascending=False).head(10))

Age                        0.163180
Spa                        0.136402
VRDeck                     0.124311
RoomService                0.119044
CryoSleep_True             0.115436
FoodCourt                  0.107472
ShoppingMall               0.095097
HomePlanet_Europa          0.022566
Cabin_G                    0.021877
Destination_TRAPPIST-1e    0.017799
dtype: float64


- Gradient Boosting

In [15]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train)
gb_acc = accuracy_score(y_test, gb.predict(X_test))
print("Gradient Boosting:", gb_acc)

Gradient Boosting: 0.8071104387291982


- Adaptive Boosting

In [18]:
from sklearn.ensemble import AdaBoostClassifier

ada = AdaBoostClassifier(random_state=42)
ada.fit(X_train, y_train)
ada_acc = accuracy_score(y_test, ada.predict(X_test))
print("AdaBoost:", ada_acc)#your code here

AdaBoost: 0.7768532526475038


Which model is the best and why?

In [20]:
** El mejor modelo es Gradient Boosting, con un accuracy de 80.7 % sobre el test, seguido muy de cerca por Random Forest (80.6 %) y Bagging (80.4 %). Los tres superan al KNN del lab anterior (≈77 %), lo que confirma la idea central de los métodos de ensemble: combinar muchos modelos simples produce un modelo más fuerte y robusto que cualquiera individual.
Gradient Boosting gana porque entrena los árboles de forma secuencial, corrigiendo en cada paso los errores del conjunto anterior. Eso le permite capturar patrones más sutiles que los métodos en paralelo (Bagging, Pasting, Random Forest), que reducen la varianza pero no afinan los errores con la misma precisión.
Resultados completos, de mejor a peor:

Gradient Boosting → 0.8071
Random Forest → 0.8056
Bagging → 0.8041
AdaBoost → 0.7769
Pasting → 0.7542 **

SyntaxError: invalid character '≈' (U+2248) (3114102209.py, line 1)